# Creating simulation replicates and baseline benchmarking

## Population size change:

In [179]:
%%bash
declare -a names_l=($(ls ../inferred_genetrees/ | grep 500Kb | grep 10X | cut -f1 -d'.' | sed 's/-500Kb-[0-9]//'))
# rm -rf ../new_replicates
mkdir -p /dev/shm/new_replicates/population_size_change
for p in 0.02 0.05 0.10 0.15 0.20 0.25; do
    for name in "${names_l[@]}"; do
        if [ ! -f /dev/shm/new_replicates/population_size_change/p${p//./}-${name}/emission.gtrees ]; then
          echo ${name}
          cat  ../inferred_genetrees/${name}-500Kb* > ../inferred_genetrees/concat-${name}.gtrees
          python ../../yule-trees/scripts/simulate_mixture_condition.py \
            -x ../inferred_genetrees/concat-default.gtrees \
            -y ../inferred_genetrees/concat-${name}.gtrees \
            -p $p -r 0.99999 -o /dev/shm/new_replicates/population_size_change/p${p//./}-${name}/
        fi
    done
done

anomaly_case-Accipitriformes_10X_up
anomaly_case-N109_10X_down
anomaly_case-N114_10X_down
anomaly_case-N118_10X_up
anomaly_case-N119_10X_down
anomaly_case-N159_10X_up
anomaly_case-N186_10X_down
anomaly_case-N220_10X_up
anomaly_case-N228_10X_down
anomaly_case-N228_10X_up
anomaly_case-N233_10X_up
anomaly_case-N238_10X_up
anomaly_case-N274_10X_down
anomaly_case-N276_10X_down
anomaly_case-N281_10X_up
anomaly_case-N299_10X_down
anomaly_case-N343_10X_down
anomaly_case-N363_10X_down
anomaly_case-N399_10X_up
anomaly_case-N425_10X_up
anomaly_case-N459_10X_down
anomaly_case-N460_10X_down
anomaly_case-N473_10X_down
anomaly_case-N491_10X_down
anomaly_case-N497_10X_up
anomaly_case-N498_10X_down
anomaly_case-N532_10X_down
anomaly_case-N544_10X_up
anomaly_case-N554_10X_up
anomaly_case-N573_10X_down
anomaly_case-N574_10X_down
anomaly_case-N575_10X_down
anomaly_case-N579_10X_up
anomaly_case-N615_10X_down
anomaly_case-N616_10X_up
anomaly_case-N617_10X_up
anomaly_case-N620_10X_up
anomaly_case-N635_10X_up

In [190]:
%%bash
declare -a names_l=($(ls ../inferred_genetrees/ | grep 500Kb | grep 10X | cut -f1 -d'.' | sed 's/-500Kb-[0-9]//' | sort | uniq))

eval "$(micromamba shell hook --shell bash)"
micromamba activate phlag

rm -f commands-population_size_change.txt
for method in pbp-hmm; do
    for name in "${names_l[@]}"; do
        for p in 0.02 0.05 0.10 0.15 0.20 0.25; do
            qqs_path="/dev/shm/new_replicates/population_size_change/p${p//./}-${name}/qqs.txt"
            edge=$(echo ${name} | cut -f2 -d '-' | cut -f1 -d'_')
            output_path=/dev/shm/new_replicates/population_size_change/p${p//./}-${name}/pred-${method//-/}_sc_ex.txt
            # if [ ! -f "$output_path" ]; then
                # echo ${output_path}
                echo "python ../../../phlag/sbag.py $method \
                     -s ../main_neoaves-num_generations.tre \
                     -g /dev/shm/new_replicates/population_size_change/p${p//./}-${name}/emission.gtrees \
                     -b ${edge} -o ${output_path} \
                     --read-qqs-freqs ${qqs_path} --expand-branches" >> commands-population_size_change.txt
            output_path=/dev/shm/new_replicates/population_size_change/p${p//./}-${name}/pred-${method//-/}_sc.txt
            # if [ ! -f "$output_path" ]; then
                # echo ${output_path}
                echo "python ../../../phlag/sbag.py $method \
                     -s ../main_neoaves-num_generations.tre \
                     -g /dev/shm/new_replicates/population_size_change/p${p//./}-${name}/emission.gtrees \
                     -b ${edge} -o ${output_path} \
                     --read-qqs-freqs ${qqs_path}" >> commands-population_size_change.txt
            # fi
        done
    done
    # wait
done

In [ ]:
%%bash
eval "$(micromamba shell hook --shell bash)"
micromamba activate phlag

declare -a names_l=($(ls ../inferred_genetrees/ | grep 500Kb | grep 10X | cut -f1 -d'.' | sed 's/-500Kb-[0-9]//' | sort | uniq))

rm -f ../results/baseline_metrics-population_size_change.tsv
for name in "${names_l[@]}"; do
    for method in gqs-hmm pbp-hmm bcb-hmm dto-hmm mlt-hmm cbp-hmm; do
        for p in 0.02 0.05 0.10 0.15 0.20 0.25; do
            metrics=$(python ../../yule-trees/scripts/compute_metrics.py \
                    -x ../replicates/population_size_change/p${p//./}-${name}/pred-${method//-/}_ex.txt \
                    -y ../replicates/population_size_change/p${p//./}-${name}/info.txt \
                    --method phlag --revert)
            if [ -n "${metrics}" ]; then
                edge=$(echo ${name} | cut -f2 -d '-' | cut -f1 -d'_')
                result=$(printf "${metrics//\\n/}\t${method}_ex\t${name}\t${p}\t${edge}")
                echo ${result} >> ../results/baseline_metrics-population_size_change.tsv
            fi
        done
    done
done

## Recombination suppression

In [191]:
%%bash
declare -a names_l=($(ls ../inferred_genetrees/ | grep 500Kb | grep recombination_suppression | cut -f1 -d'.' | sed 's/-500Kb-[0-9]//' | sort | uniq))

mkdir -p /dev/shm/new_replicates/recombination_suppression

for name in "${names_l[@]}"; do
    # cat  ../inferred_genetrees/${name}-500Kb* > ../inferred_genetrees/concat-${name}.gtrees
    for p in 0.02 0.05 0.10 0.15 0.20 0.25; do
        # if [ ! -f /dev/shm/new_replicates/recombination_suppression/p${p//./}-${name}/emission.gtrees ]; then
            echo ${name} ${p}
            edge=$(echo ${name} | cut -f3 -d'-')
            if grep -w -q "${edge}" ../main_neoaves-num_generations.tre; then
                default_gtrees="../inferred_genetrees/concat-default.gtrees"
            else
                default_gtrees="../inferred_genetrees/concat-favian.gtrees"
            fi
            python ../../yule-trees/scripts/simulate_mixture_condition.py \
                -x ${default_gtrees} \
                -y ../inferred_genetrees/concat-${name}.gtrees \
                -p $p -r 0.99999 -o /dev/shm/new_replicates/recombination_suppression/p${p//./}-${name}/
        # fi
    done
done

anomaly_case-recombination_suppression-N115 0.02
anomaly_case-recombination_suppression-N115 0.05
anomaly_case-recombination_suppression-N115 0.10
anomaly_case-recombination_suppression-N115 0.15
anomaly_case-recombination_suppression-N115 0.20
anomaly_case-recombination_suppression-N115 0.25
anomaly_case-recombination_suppression-N125 0.02
anomaly_case-recombination_suppression-N125 0.05
anomaly_case-recombination_suppression-N125 0.10
anomaly_case-recombination_suppression-N125 0.15
anomaly_case-recombination_suppression-N125 0.20
anomaly_case-recombination_suppression-N125 0.25
anomaly_case-recombination_suppression-N202 0.02
anomaly_case-recombination_suppression-N202 0.05
anomaly_case-recombination_suppression-N202 0.10
anomaly_case-recombination_suppression-N202 0.15
anomaly_case-recombination_suppression-N202 0.20
anomaly_case-recombination_suppression-N202 0.25
anomaly_case-recombination_suppression-N211 0.02
anomaly_case-recombination_suppression-N211 0.05
anomaly_case-recombi

In [ ]:
%%bash
declare -a names_l=($(ls ../inferred_genetrees/ | grep 500Kb | grep recombination_suppression | cut -f1 -d'.' | sed 's/-500Kb-[0-9]//' | sort | uniq))

eval "$(micromamba shell hook --shell bash)"
micromamba activate phlag

rm -f commands-recombination_suppression.txt
for name in "${names_l[@]}"; do
    for method in cqs-hmm; do
        for p in 0.02 0.05 0.10 0.15 0.20 0.25; do
            edge=$(echo ${name} | cut -f3 -d'-')
            qqs_path="../replicates/recombination_suppression/p${p//./}-${name}/qqs.txt"
            output_path=../replicates/recombination_suppression/p${p//./}-${name}/pred-${method//-/}_ilr_ex.txt
            # if [ ! -f "${output_path}" ]; then
                # echo ${output_path}
                if grep -q -w -E "N36|N722|N71" <<< "${edge}" ; then
                    species_tree="../main-num_generations.tre"
                else
                    species_tree="../main_neoaves-num_generations.tre"
                fi
                echo "python ../../../phlag/sbag.py $method \
                     -s ${species_tree} \
                     -g ../replicates/recombination_suppression/p${p//./}-${name}/emission.gtrees \
                     -b ${edge} -o ${output_path} \
                     --read-qqs-freqs ${qqs_path} --transform-ilr --expand-branches" >> commands-recombination_suppression.txt
            # fi
            output_path=../replicates/recombination_suppression/p${p//./}-${name}/pred-${method//-/}_ex.txt
            # if [ ! -f "${output_path}" ]; then
                # echo ${output_path}
                if grep -q -w -E "N36|N722|N71" <<< "${edge}" ; then
                    species_tree="../main-num_generations.tre"
                else
                    species_tree="../main_neoaves-num_generations.tre"
                fi
                echo "python ../../../phlag/sbag.py $method \
                     -s ${species_tree} \
                     -g ../replicates/recombination_suppression/p${p//./}-${name}/emission.gtrees \
                     -b ${edge} -o ${output_path} \
                     --read-qqs-freqs ${qqs_path} --expand-branches" >> commands-recombination_suppression.txt
            # fi
        done
    done
    # echo "Waiting for ${name}"
    # wait
done

In [ ]:
%%bash
eval "$(micromamba shell hook --shell bash)"
micromamba activate phlag

declare -a names_l=($(ls ../inferred_genetrees/ | grep 500Kb | grep recombination_suppression | cut -f1 -d'.' | sed 's/-500Kb-[0-9]//' | sort | uniq))

rm -f ../results/baseline_metrics-recombination_suppression.tsv
for name in "${names_l[@]}"; do
    for method in gqs-hmm pbp-hmm bcb-hmm dto-hmm mlt-hmm; do
        for p in 0.02 0.05 0.10 0.15 0.20 0.25; do
            echo $name $method $p
            metrics=$(python ../../yule-trees/scripts/compute_metrics.py \
                    -x ../replicates/recombination_suppression/p${p//./}-${name}/pred-${method//-/}_ex.txt \
                    -y ../replicates/recombination_suppression/p${p//./}-${name}/info.txt \
                    --method phlag --revert)
            if [ -n "${metrics}" ]; then
                edge=$(echo "${name}" | cut -f3 -d'-')
                result=$(printf "${metrics//\\n/}\t${method}_ex\t${name}\t${p}\t${edge}")
                echo ${result} >> ../results/baseline_ex_metrics-recombination_suppression.tsv
            fi
        done
    done
done

## Recombination increase

In [ ]:
%%bash
declare -a names_l=($(ls ../inferred_genetrees/ | grep 500Kb | grep recombination_increase | cut -f1 -d'.' | sed 's/-500Kb-[0-9]//' | sort | uniq))

mkdir -p ../replicates/recombination_increase

for name in "${names_l[@]}"; do
    cat  ../inferred_genetrees/${name}-500Kb* > ../inferred_genetrees/concat-${name}.gtrees
    for p in 0.02 0.05 0.10 0.15 0.20 0.25; do
        # if [ ! -f ../replicates/recombination_increase/p${p//./}-${name}/emission.gtrees ]; then
            echo ${name} ${p}
            edge=$(echo ${name} | cut -f3 -d'-')
            if grep -w -q "${edge}" ../main_neoaves-num_generations.tre; then
                default_gtrees="../inferred_genetrees/concat-default.gtrees"
            else
                default_gtrees="../inferred_genetrees/concat-favian.gtrees"
            fi
            python ../../yule-trees/scripts/simulate_mixture_condition.py \
                -x ${default_gtrees} \
                -y ../inferred_genetrees/concat-${name}.gtrees \
                -p $p -r 0.99999 -o ../replicates/recombination_increase/p${p//./}-${name}/
        # fi
    done
done

In [ ]:
%%bash
declare -a names_l=($(ls ../inferred_genetrees/ | grep 500Kb | grep recombination_increase | cut -f1 -d'.' | sed 's/-500Kb-[0-9]//' | sort | uniq))

eval "$(micromamba shell hook --shell bash)"
micromamba activate phlag

rm -f commands-recombination_increase.txt
for name in "${names_l[@]}"; do
    for method in bcb-hmm; do
        for p in 0.02 0.05 0.10 0.15 0.20 0.25; do
            edge=$(echo ${name} | cut -f3 -d'-')
            qqs_path="../replicates/recombination_increase/p${p//./}-${name}/qqs.txt"
            output_path=../replicates/recombination_increase/p${p//./}-${name}/pred-${method//-/}_sc_ex.txt
            # if [ ! -f "${output_path}" ]; then
                # echo ${output_path}
                if grep -q -w -E "N36|N722|N71" <<< "${edge}" ; then
                    species_tree="../main-num_generations.tre"
                else
                    species_tree="../main_neoaves-num_generations.tre"
                fi
                echo "python ../../../phlag/sbag.py $method \
                     -s ${species_tree}\
                     -g ../replicates/recombination_increase/p${p//./}-${name}/emission.gtrees \
                     -b ${edge} -o ${output_path} \
                     --read-qqs-freqs ${qqs_path} --expand-branches" >> commands-recombination_increase.txt
            output_path=../replicates/recombination_increase/p${p//./}-${name}/pred-${method//-/}_sc.txt
            # if [ ! -f "${output_path}" ]; then
                # echo ${output_path}
                if grep -q -w -E "N36|N722|N71" <<< "${edge}" ; then
                    species_tree="../main-num_generations.tre"
                else
                    species_tree="../main_neoaves-num_generations.tre"
                fi
                echo "python ../../../phlag/sbag.py $method \
                     -s ${species_tree}\
                     -g ../replicates/recombination_increase/p${p//./}-${name}/emission.gtrees \
                     -b ${edge} -o ${output_path} \
                     --read-qqs-freqs ${qqs_path}" >> commands-recombination_increase.txt
            # fi
        done
    done
    # echo "Waiting for ${name}"
    # wait
done

In [ ]:
%%bash
eval "$(micromamba shell hook --shell bash)"
micromamba activate phlag

declare -a names_l=($(ls ../inferred_genetrees/ | grep 500Kb | grep recombination_increase | cut -f1 -d'.' | sed 's/-500Kb-[0-9]//' | sort | uniq))

rm -f ../results/baseline_metrics-recombination_increase.tsv
for name in "${names_l[@]}"; do
    for method in gqs-hmm pbp-hmm bcb-hmm dto-hmm mlt-hmm; do
        for p in 0.02 0.05 0.10 0.15 0.20 0.25; do
            echo $name $method $p
            metrics=$(python ../../yule-trees/scripts/compute_metrics.py \
                    -x ../replicates/recombination_increase/p${p//./}-${name}/pred-${method//-/}_ex.txt \
                    -y ../replicates/recombination_increase/p${p//./}-${name}/info.txt \
                    --method phlag --revert)
            if [ -n "${metrics}" ]; then
                edge=$(echo "${name}_ex" | cut -f3 -d'-')
                result=$(printf "${metrics//\\n/}\t${method}\t${name}\t${p}\t${edge}")
                echo ${result} >> ../results/baseline_ex_metrics-recombination_increase.tsv
            fi
        done
    done
done

## Admixture

In [138]:
%%bash
declare -a names_l=($(ls /dev/shm/inferred_genetrees/ | grep 500Kb | grep admixture | cut -f1 -d'.' | sed 's/-500Kb-[0-9]//' | sort | uniq | grep rate | grep time))

mkdir -p ../replicates/admixture

for name in "${names_l[@]}"; do
    cat  /dev/shm/inferred_genetrees/${name}-500Kb* > ../inferred_genetrees/concat-${name}.gtrees
    for p in 0.02 0.05 0.10 0.15 0.20 0.25; do
        # if [ ! -f ../replicates/recombination_increase/p${p//./}-${name}/emission.gtrees ]; then
            echo ${name} ${p}
            edge=$(echo ${name} | cut -f3 -d'-')
            default_gtrees="../inferred_genetrees/concat-default.gtrees"
            python ../../yule-trees/scripts/simulate_mixture_condition.py \
                 -x ${default_gtrees} \
                -y ../inferred_genetrees/concat-${name}.gtrees \
                -p $p -r 0.99999 -o ../replicates/admixture/p${p//./}-${name}/
        # fi
    done
done

anomaly_case-admixture-Cariamiformes_Falconiformes_rate090_time6099554 0.02
anomaly_case-admixture-Cariamiformes_Falconiformes_rate090_time6099554 0.05
anomaly_case-admixture-Cariamiformes_Falconiformes_rate090_time6099554 0.10
anomaly_case-admixture-Cariamiformes_Falconiformes_rate090_time6099554 0.15
anomaly_case-admixture-Cariamiformes_Falconiformes_rate090_time6099554 0.20
anomaly_case-admixture-Cariamiformes_Falconiformes_rate090_time6099554 0.25
anomaly_case-admixture-Columbiformes_Mesitornisunicolor_rate090_time6066340 0.02
anomaly_case-admixture-Columbiformes_Mesitornisunicolor_rate090_time6066340 0.05
anomaly_case-admixture-Columbiformes_Mesitornisunicolor_rate090_time6066340 0.10
anomaly_case-admixture-Columbiformes_Mesitornisunicolor_rate090_time6066340 0.15
anomaly_case-admixture-Columbiformes_Mesitornisunicolor_rate090_time6066340 0.20
anomaly_case-admixture-Columbiformes_Mesitornisunicolor_rate090_time6066340 0.25
anomaly_case-admixture-N340_Eubuccobourcierii_rate090_time

In [215]:
%%bash
declare -a names_l=($(ls /dev/shm/inferred_genetrees/ | grep 500Kb | grep admixture | cut -f1 -d'.' \
    | grep rate | grep time  | sed 's/-500Kb-[0-9]//' | sort | uniq))

eval "$(micromamba shell hook --shell bash)"
micromamba activate phlag

rm -f commands-admixture.txt
method="phlag_dto_eap05_ena4_esp001"
options="--expected-num-anomalies 4  --expected-anamoly-proportion 0.50"
# for method in gqs-hmm cqs-hmm pbp-hmm bcb-hmm dto-hmm mlt-hmm; do
    for name in "${names_l[@]}"; do
        for p in 0.02 0.05 0.10 0.15 0.20 0.25; do
            echo $name
            qqs_path="../replicates/admixture/p${p//./}-${name}/qqs.txt"
            
            edge_derived=$(echo "${name}" | cut -f3 -d'-' | cut -f1 -d'_')
            if [[ "$edge_derived" == "Cariamiformes" ]]; then
                edge="N718"
            fi
            if [[ "$edge_derived" == "N340" ]]; then
                edge="N341"
            fi
            if [[ "$edge_derived" == "N367" ]]; then
                edge="N368"
            fi
            if [[ "$edge_derived" == "N482" ]]; then
                edge="N494"
            fi
            if [[ "$edge_derived" == "N491" ]]; then
                edge="N492"
            fi
            if [[ "$edge_derived" == "N555" ]]; then
                edge="N567"
            fi
            if [[ "$edge_derived" == "N560" ]]; then
                edge="N566"
            fi
            if [[ "$edge_derived" == "N576" ]]; then
                edge="N704"
            fi
            if [[ "$edge_derived" == "N90" ]]; then
                edge="N91"
            fi
            if [[ "$edge_derived" == "Otidiformes" ]]; then
                edge="N117"
            fi
            if [[ "$edge_derived" == "Strigiformes" ]]; then
                edge="N299"
            fi
            if [[ "$edge_derived" == "Trogoniformes" ]]; then
                edge="N348"
            fi
            if [[ "$edge_derived" == "Columbiformes" ]]; then
                edge="N93"
            fi
            output_path="../replicates/admixture/p${p//./}-${name}/pred-${method//-/}_ex.txt"
            echo "python ../../../phlag/phlag.py \
             -s ../main_neoaves-num_generations.tre \
             -g ../replicates/admixture/p${p//./}-${name}/emission.gtrees \
             -c ${edge} -o ${output_path} \
             --read-qqs-freqs ${qqs_path} ${options} --expand-branches" >> commands-admixture.txt
            
            output_path="../replicates/admixture/p${p//./}-${name}/pred-${method//-/}.txt"
            echo "python ../../../phlag/phlag.py \
             -s ../main_neoaves-num_generations.tre \
             -g ../replicates/admixture/p${p//./}-${name}/emission.gtrees \
             -c ${edge} -o ${output_path} \
             --read-qqs-freqs ${qqs_path}  ${options}" >> commands-admixture.txt
            #output_path=../replicates/admixture/p${p//./}-${name}/pred-${method//-/}_ex.txt
            # if [ ! -f "$output_path" ]; then
                # echo ${output_path}
            #    echo "python ../../../phlag/sbag.py $method \
            #         -s ../main_neoaves-num_generations.tre \
            #         -g ../replicates/admixture/p${p//./}-${name}/emission.gtrees \
            #         -b ${edge} -o ${output_path} \
            #         --read-qqs-freqs ${qqs_path} --expand-branches" >> commands-admixture.txt
            # output_path=../replicates/admixture/p${p//./}-${name}/pred-${method//-/}.txt
            # if [ ! -f "$output_path" ]; then
            #    # echo ${output_path}
            #    echo "python ../../../phlag/sbag.py $method \
            #         -s ../main_neoaves-num_generations.tre \
            #         -g ../replicates/admixture/p${p//./}-${name}/emission.gtrees \
            #         -b ${edge} -o ${output_path} \
            #         --read-qqs-freqs ${qqs_path}" >> commands-admixture.txt
            # fi
        done
    done
    # wait
# done

anomaly_case-admixture-Cariamiformes_Falconiformes_rate090_time6099554
anomaly_case-admixture-Cariamiformes_Falconiformes_rate090_time6099554
anomaly_case-admixture-Cariamiformes_Falconiformes_rate090_time6099554
anomaly_case-admixture-Cariamiformes_Falconiformes_rate090_time6099554
anomaly_case-admixture-Cariamiformes_Falconiformes_rate090_time6099554
anomaly_case-admixture-Cariamiformes_Falconiformes_rate090_time6099554
anomaly_case-admixture-Columbiformes_Mesitornisunicolor_rate090_time6066340
anomaly_case-admixture-Columbiformes_Mesitornisunicolor_rate090_time6066340
anomaly_case-admixture-Columbiformes_Mesitornisunicolor_rate090_time6066340
anomaly_case-admixture-Columbiformes_Mesitornisunicolor_rate090_time6066340
anomaly_case-admixture-Columbiformes_Mesitornisunicolor_rate090_time6066340
anomaly_case-admixture-Columbiformes_Mesitornisunicolor_rate090_time6066340
anomaly_case-admixture-N340_Eubuccobourcierii_rate090_time1599554
anomaly_case-admixture-N340_Eubuccobourcierii_rate09

In [151]:
%%bash
declare -a names_l=($(ls /dev/shm//inferred_genetrees/ | grep 500Kb | grep admixture | cut -f1 -d'.' | sed 's/-500Kb-[0-9]//' | sort | uniq))

eval "$(micromamba shell hook --shell bash)"
micromamba activate phlag

rm -f commands-admixture.txt
# cqs-hmm pbp-hmm bcb-hmm dto-hmm mlt-hmm
for method in gqs-hmm; do
    for name in "${names_l[@]}"; do
        for p in 0.02 0.05 0.10 0.15 0.20 0.25; do
            qqs_path="../replicates/admixture/p${p//./}-${name}/qqs.txt"
            edge1=$(echo "${name}" | cut -f3 -d'-' | cut -f6 -d'_')
            if [[ "$edge1" == "Opisthocomushoazin" ]]; then
                edge1="N192"
            fi
            edge2=$(echo "${name}" | cut -f3 -d'-' | cut -f2 -d'_' | sed 's/_/ /')
            if [[ "$edge2" == "Mesitornisunicolor" ]]; then
                edge2="N83"
            fi
            edge3=$(echo "${name}" | cut -f3 -d'-' | cut -f4 -d'_' | sed 's/_/ /')
            if [[ "$edge3" == "Mesitornisunicolor" ]]; then
                edge3="N83"
            fi
            edge="${edge1}"
            output_path=../replicates/admixture/p${p//./}-${name}/pred-${method//-/}_ex_derived.txt
            # if [ ! -f "$output_path" ]; then
                # echo ${output_path}
                echo "python ../../../phlag/sbag.py $method \
                     -s ../main_neoaves-num_generations.tre \
                     -g ../replicates/admixture/p${p//./}-${name}/emission.gtrees \
                     -b ${edge} -o ${output_path} \
                     --write-qqs-freqs ${qqs_path} --expand-branches" >> commands-admixture.txt
            output_path=../replicates/admixture/p${p//./}-${name}/pred-${method//-/}_derived.txt
            # if [ ! -f "$output_path" ]; then
                # echo ${output_path}
                echo "python ../../../phlag/sbag.py $method \
                     -s ../main_neoaves-num_generations.tre \
                     -g ../replicates/admixture/p${p//./}-${name}/emission.gtrees \
                     -b ${edge} -o ${output_path} \
                     --write-qqs-freqs ${qqs_path}" >> commands-admixture.txt
            # fi
        done
    done
    # wait
done

Process is interrupted.


In [226]:
%%bash
eval "$(micromamba shell hook --shell bash)"
micromamba activate phlag

declare -a paths_l=($(find ../replicates/admixture -name "*pred*" |grep "phlag_dto_.*eap001.*_esp001.*"))
rm -f commands-metrics.txt
for path in "${paths_l[@]}"; do
    echo $path
    name=$(echo ${path} | xargs -I{} basename {} | cut -f2 -d'-' | sed 's/.txt//')
    dirpath=$(echo ${path} | xargs -I{} dirname {})
    edge=$(echo "${path}" | cut -f4 -d'-' | cut -f1 -d'_')
    time=$(echo "${path}" | cut -f4 -d'-' | cut -f4 -d'_' | sed 's/time//'  | sed 's/\/pred//')
    rate=$(echo "${path}" | cut -f4 -d'-' | cut -f3 -d'_' | sed 's/rate//')
    echo "python ../../yule-trees/scripts/compute_metrics.py \
            -x ${path} \
            -y ${dirpath}/info.txt \
            --method phlag --revert \
            --labels admixture ${name} ${edge} ${time} ${rate} \
           > ${dirpath}/metrics-${name}.txt" >> commands-metrics.txt
done

../replicates/admixture/p005-anomaly_case-admixture-Cariamiformes_Falconiformes_rate090_time6099554/pred-phlag_dto_eap001_ena4_esp001.txt
../replicates/admixture/p005-anomaly_case-admixture-Cariamiformes_Falconiformes_rate090_time6099554/pred-phlag_dto_eap001_ena4_esp001_ex.txt
../replicates/admixture/p015-anomaly_case-admixture-N560_N565_rate090_time1599554/pred-phlag_dto_eap001_ena4_esp001.txt
../replicates/admixture/p015-anomaly_case-admixture-N560_N565_rate090_time1599554/pred-phlag_dto_eap001_ena4_esp001_ex.txt
../replicates/admixture/p020-anomaly_case-admixture-Columbiformes_Mesitornisunicolor_rate090_time6066340/pred-phlag_dto_eap001_ena4_esp001.txt
../replicates/admixture/p020-anomaly_case-admixture-Columbiformes_Mesitornisunicolor_rate090_time6066340/pred-phlag_dto_eap001_ena4_esp001_ex.txt
../replicates/admixture/p025-anomaly_case-admixture-Cariamiformes_Falconiformes_rate090_time6099554/pred-phlag_dto_eap001_ena4_esp001.txt
../replicates/admixture/p025-anomaly_case-admixture

In [228]:
%%bash
eval "$(micromamba shell hook --shell bash)"
micromamba activate phlag

declare -a paths_l=($(find ../replicates/population_size_change/ -name "*pred*" | grep "pred-phlag_dto_eap01_ena4_esp001" | grep -v noprior))
# rm -f commands-metrics.txt
for path in "${paths_l[@]}"; do
    echo $path
    dirpath=$(echo ${path} | xargs -I{} dirname {})
    edge=$(echo $dirpath | cut -d'-' -f3 | cut -d'_' -f1)
    type=$(echo $dirpath | cut -d'-' -f3 | cut -d'_' -f2,3)
    name=$(echo ${path} | xargs -I{} basename {} | cut -f2 -d'-' | sed 's/.txt//')
    output_path="$(echo ${path} | xargs -I{} dirname {})/metrics.txt"
    echo "python ../../yule-trees/scripts/compute_metrics.py \
            -x ${path} \
            -y ${dirpath}/info.txt \
            --method phlag --revert \
            --labels population_size_change ${name} ${edge} ${type} \
           > ${dirpath}/metrics-${name}.txt" >> commands-metrics.txt
done

../replicates/population_size_change/p002-anomaly_case-N615_10X_down/pred-phlag_dto_eap01_ena4_esp001.txt
../replicates/population_size_change/p002-anomaly_case-N615_10X_down/pred-phlag_dto_eap01_ena4_esp001_ex.txt
../replicates/population_size_change/p025-anomaly_case-N491_10X_down/pred-phlag_dto_eap01_ena4_esp001.txt
../replicates/population_size_change/p025-anomaly_case-N491_10X_down/pred-phlag_dto_eap01_ena4_esp001_ex.txt
../replicates/population_size_change/p015-anomaly_case-N473_10X_down/pred-phlag_dto_eap01_ena4_esp001.txt
../replicates/population_size_change/p015-anomaly_case-N473_10X_down/pred-phlag_dto_eap01_ena4_esp001_ex.txt
../replicates/population_size_change/p002-anomaly_case-N616_10X_up/pred-phlag_dto_eap01_ena4_esp001.txt
../replicates/population_size_change/p002-anomaly_case-N616_10X_up/pred-phlag_dto_eap01_ena4_esp001_ex.txt
../replicates/population_size_change/p010-anomaly_case-N579_10X_up/pred-phlag_dto_eap01_ena4_esp001.txt
../replicates/population_size_change/p01

In [229]:
%%bash
eval "$(micromamba shell hook --shell bash)"
micromamba activate phlag

declare -a paths_l=($(find ../replicates/recombination_suppression/ -name "*pred*" | grep "pred-phlag_dto_eap01_ena4_esp001" | grep -v noprior))
# rm -f commands-metrics.txt
for path in "${paths_l[@]}"; do
    echo $path
    dirpath=$(echo ${path} | xargs -I{} dirname {})
    edge=$(echo $dirpath | cut -d'-' -f4 | cut -d'_' -f1)
    type=$(echo $dirpath | cut -d'-' -f3 )
    name=$(echo ${path} | xargs -I{} basename {} | cut -f2 -d'-' | sed 's/.txt//')
    output_path="$(echo ${path} | xargs -I{} dirname {})/metrics.txt"
    echo "python ../../yule-trees/scripts/compute_metrics.py \
            -x ${path} \
            -y ${dirpath}/info.txt \
            --method phlag --revert \
            --labels recombination ${name} ${edge} ${type} \
           > ${dirpath}/metrics-${name}.txt" >> commands-metrics.txt
done

../replicates/recombination_suppression/p010-anomaly_case-recombination_suppression-N717/pred-phlag_dto_eap01_ena4_esp001.txt
../replicates/recombination_suppression/p010-anomaly_case-recombination_suppression-N717/pred-phlag_dto_eap01_ena4_esp001_ex.txt
../replicates/recombination_suppression/p002-anomaly_case-recombination_suppression-N396/pred-phlag_dto_eap01_ena4_esp001.txt
../replicates/recombination_suppression/p002-anomaly_case-recombination_suppression-N396/pred-phlag_dto_eap01_ena4_esp001_ex.txt
../replicates/recombination_suppression/p025-anomaly_case-recombination_suppression-N533/pred-phlag_dto_eap01_ena4_esp001.txt
../replicates/recombination_suppression/p025-anomaly_case-recombination_suppression-N533/pred-phlag_dto_eap01_ena4_esp001_ex.txt
../replicates/recombination_suppression/p020-anomaly_case-recombination_suppression-N396/pred-phlag_dto_eap01_ena4_esp001.txt
../replicates/recombination_suppression/p020-anomaly_case-recombination_suppression-N396/pred-phlag_dto_eap01

In [230]:

%%bash
eval "$(micromamba shell hook --shell bash)"
micromamba activate phlag

declare -a paths_l=($(find ../replicates/recombination_increase/ -name "*pred*" | grep "pred-phlag_dto_eap01_ena4_esp001" | grep -v noprior))
# rm -f commands-metrics.txt
for path in "${paths_l[@]}"; do
    echo $path
    dirpath=$(echo ${path} | xargs -I{} dirname {})
    edge=$(echo $dirpath | cut -d'-' -f4 | cut -d'_' -f1)
    type=$(echo $dirpath | cut -d'-' -f3 )
    name=$(echo ${path} | xargs -I{} basename {} | cut -f2 -d'-' | sed 's/.txt//')
    output_path="$(echo ${path} | xargs -I{} dirname {})/metrics.txt"
    echo "python ../../yule-trees/scripts/compute_metrics.py \
            -x ${path} \
            -y ${dirpath}/info.txt \
            --method phlag --revert \
            --labels recombination ${name} ${edge} ${type} \
           > ${dirpath}/metrics-${name}.txt" >> commands-metrics.txt
done

../replicates/recombination_increase/p025-anomaly_case-recombination_increase-N461/pred-phlag_dto_eap01_ena4_esp001.txt
../replicates/recombination_increase/p025-anomaly_case-recombination_increase-N461/pred-phlag_dto_eap01_ena4_esp001_ex.txt
../replicates/recombination_increase/p020-anomaly_case-recombination_increase-N238/pred-phlag_dto_eap01_ena4_esp001.txt
../replicates/recombination_increase/p020-anomaly_case-recombination_increase-N238/pred-phlag_dto_eap01_ena4_esp001_ex.txt
../replicates/recombination_increase/p005-anomaly_case-recombination_increase-N716/pred-phlag_dto_eap01_ena4_esp001.txt
../replicates/recombination_increase/p005-anomaly_case-recombination_increase-N716/pred-phlag_dto_eap01_ena4_esp001_ex.txt
../replicates/recombination_increase/p025-anomaly_case-recombination_increase-N151/pred-phlag_dto_eap01_ena4_esp001.txt
../replicates/recombination_increase/p025-anomaly_case-recombination_increase-N151/pred-phlag_dto_eap01_ena4_esp001_ex.txt
../replicates/recombination_